# Experiment 3 — GM-MolSG vs GE-MolSG vs shape/ESP baselines (DUDE-Z)

Per-target retrieval (EF1%, BEDROC α=20) for six methods:

- **GM-MolSG** — fused descriptor: geo WKS BoF **+** surface-patch VLAD, combined
  by linear similarity fusion `S = alpha·S_geo + (1-alpha)·S_vlad`. 
- **GE-MolSG** — standalone geo WKS → hard k-NN BoF, chi-squared retrieval.
- **ElectroShape** — 4D ElectroShape (oddt), MMFF94 charges, USR similarity.
- **ElectroShape5D** — 5D ElectroShape (Armstrong 2011), MMFF94 charges.
- **ESP-Sim** — Crippen-O3A + shape + ML-charge ESP (espsim).
- **Roshambo** — Roshambo2 GPU shape/ESP overlay (combination score).

Same workflow as experiments 1–2: iterative per-target generation with on-disk
caching, summary, two scatter plots (BEDROC, EF1%), and a Wilcoxon test focused
on GM-MolSG. Each method is logged if its libraries are unavailable.

### Inputs
```
<DATA_ROOT>/DUDE-Z/<TARGET>.tar.gz -> <TARGET>/ESP_Npy/{ligand,decoy}_<ID>.npy
                                       <TARGET>/PDB_Files/<ID>.pdb
experiments/codebooks/ge_molsg_cb.npy        (GE-MolSG geo codebook)
experiments/codebooks/gm_geo_cb.npy          (GM-MolSG geo codebook)
experiments/codebooks/gm_patch_cb.npy        (GM-MolSG patch VLAD codebook)
<QUERIES_DIR>/<target>.csv                   (column 'query' or 'queries')
```


## 1 · Configuration

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import logging, sys, tarfile
from pathlib import Path
import numpy as np
import pandas as pd

# Directory holding the bundled experiment modules (exp3_methods.py, vlad.py,
# patches.py). Set EXP_DIR explicitly if you run this notebook from elsewhere;
# by default it searches the CWD and its scripts/experiments subdir, then walks
# up parent directories.
EXP_DIR = None  # e.g. Path("/abs/path/to/repo/scripts/experiments")
def _find_exp_dir(explicit):
    if explicit is not None:
        return Path(explicit).resolve()
    here = Path.cwd().resolve()
    cands = [here / "scripts" / "experiments", here]
    cands += [p / "scripts" / "experiments" for p in here.parents]
    cands += [here / "experiments"]
    for cand in cands:
        if (cand / "exp3_methods.py").exists():
            return cand.resolve()
    raise FileNotFoundError(
        "Could not locate exp3_methods.py. Set EXP_DIR to scripts/experiments.")
EXP_DIR = _find_exp_dir(EXP_DIR)
sys.path.insert(0, str(EXP_DIR))   # bundled: exp3_methods, vlad, patches
log_exp_dir = EXP_DIR

TARGETS = [
    "AA2AR", "ABL1", "ACES", "ADA", "ADRB2", "AMPC", "ANDR", "CSF1R",
    "CXCR4", "DEF", "DRD4", "EGFR", "FA10", "FA7", "FABP4", "FGFR1",
    "FKB1A", "GLCM", "HDAC8", "HIVPR", "HMDH", "HS90A", "ITAL", "KIT",
    "KITH", "LCK", "MAPK2", "MK01", "MT1", "NRAM", "PARP1", "PLK1",
    "PPARA", "PTN1", "PUR2", "RENI", "ROCK1", "SRC", "THRB", "TRY1",
    "TRYB1", "UROK", "XIAP",
]  # comment out any targets you don't want to run
# Data: per-target archives <TARGET>.tar.gz are hosted in one Zenodo record.
# Set ZENODO_BASE_URL and each listed target is downloaded + extracted on demand
# (only the targets in TARGETS are fetched). Each <TARGET>.tar.gz extracts to
# <TARGET>/{ESP_Npy, ESP_Npy_MMFF94, PDB_Files}/.
ZENODO_BASE_URL = "https://zenodo.org/records/20547837/files"   # e.g. "https://zenodo.org/records/XXXXXXX/files"
DATA_ROOT    = Path("dude_z_data")   # local cache for downloaded/extracted targets
QUERIES_DIR  = Path("queries")
OUT_DIR      = Path("experiments_out/exp3_gmmolsg_baselines")
CODEBOOK_DIR = Path("codebooks")

GE_CODEBOOK       = CODEBOOK_DIR / "ge_molsg_cb.npy"   # GE-MolSG geo
GM_GEO_CODEBOOK   = CODEBOOK_DIR / "gm_geo_cb.npy"     # GM-MolSG geo
GM_PATCH_CODEBOOK = CODEBOOK_DIR / "gm_patch_cb.npy"   # GM-MolSG patch VLAD

METHODS = ["GM-MolSG", "GE-MolSG", "ElectroShape", "ElectroShape5D", "ESP-Sim", "Roshambo"]
METRICS = ["EF1%", "BEDROC"]

# Shared geo / WKS params
K, EVALS, VAR, KNN, EW, LAP_NORM, BOF_KNN = 100, 100, 15, 100, 0.3, "normalized", 3

# GM-MolSG patch-VLAD + fusion
PATCH_FTS    = "esp,logp,sa,sdc,sdx,gauss_curv,mean_curv,si"
PATCH_RADII  = [2.5]
PATCH_NORM   = True
ALPHA        = 0.35           # geo weight in linear fusion
LOGP_SIGMA, LOGP_RADIUS = 2.5, 6.0
JAZZY_SIGMA, JAZZY_RADIUS = 2.5, 6.0

ESHAPE_CHARGE_MODEL = "mmff94"

FORCE = False

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT = OUT_DIR / "results"
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(OUT_DIR / "exp3.log")],
    force=True)
log = logging.getLogger("exp3")
log.info("Experiment 3 — methods: %s", ", ".join(METHODS))

## 2 · Data / query helpers

In [ ]:
import subprocess
import urllib.request


def fetch_target(target, data_root):
    """Ensure <data_root>/<TARGET>/ exists, downloading from Zenodo if needed.

    Looks for an already-extracted ``<TARGET>/ESP_Npy`` first. If absent and
    ``ZENODO_BASE_URL`` is set, downloads ``<ZENODO_BASE_URL>/<TARGET>.tar.gz``
    and extracts it under ``data_root``. Each archive extracts to
    ``<TARGET>/{ESP_Npy, ESP_Npy_MMFF94, PDB_Files}/``.

    Returns the ``<TARGET>/`` directory.
    """
    data_root = Path(data_root)
    for cand in (data_root / target, data_root / target.upper(),
                 data_root / "DUDE-Z" / target):
        if (cand / "ESP_Npy").is_dir():
            return cand

    local_tar = None
    for tar in (data_root / f"{target}.tar.gz",
                data_root / "DUDE-Z" / f"{target}.tar.gz"):
        if tar.exists():
            local_tar = tar
            break

    if local_tar is None:
        if not ZENODO_BASE_URL:
            raise FileNotFoundError(
                f"No local data for '{target}' and ZENODO_BASE_URL is not set.")
        data_root.mkdir(parents=True, exist_ok=True)
        local_tar = data_root / f"{target}.tar.gz"
        url = f"{ZENODO_BASE_URL.rstrip('/')}/{target}.tar.gz"
        log.info("Downloading %s", url)
        try:
            subprocess.run(["curl", "-fSL", "-o", str(local_tar), url], check=True)
        except (FileNotFoundError, subprocess.CalledProcessError):
            urllib.request.urlretrieve(url, local_tar)

    log.info("Extracting %s", local_tar)
    with tarfile.open(local_tar, "r:gz") as tf:
        tf.extractall(data_root)

    for cand in (data_root / target, data_root / target.upper()):
        if (cand / "ESP_Npy").is_dir():
            return cand
    hits = list(data_root.rglob(f"{target}/ESP_Npy")) or list(data_root.rglob("ESP_Npy"))
    if hits:
        return hits[0].parent
    raise FileNotFoundError(f"Extracted '{target}' but found no ESP_Npy under {data_root}")


def resolve_target_dir(target, data_root, work=None):
    """Return (esp_npy_dir, target_root). target_root also holds
    ESP_Npy_MMFF94 and PDB_Files."""
    target_root = fetch_target(target, data_root)
    esp_dir = target_root / "ESP_Npy"
    if not esp_dir.is_dir():
        raise FileNotFoundError(f"No ESP_Npy for '{target}' under {target_root}")
    return esp_dir, target_root

def resolve_queries_csv(queries_dir, target):
    for name in (f"{target}.csv", f"{target.lower()}.csv", f"{target.upper()}.csv"):
        if (queries_dir / name).exists():
            return queries_dir / name
    raise FileNotFoundError(f"No queries CSV for '{target}' under {queries_dir}")

def load_query_ids(csv_path):
    df = pd.read_csv(csv_path)
    col = next((c for c in ("queries", "query") if c in df.columns), None)
    if col is None:
        col = df.columns[0] if df.shape[1] == 1 else None
    if col is None:
        raise ValueError(f"{csv_path}: no 'query'/'queries' column found")
    return [str(q).strip() for q in df[col].dropna()]

def list_surface_fns(surf_dir):
    return sorted([f for f in os.listdir(surf_dir) if f.endswith(".npy")], reverse=True)

## 3 · Retrieval metrics (generic sim_fn)

In [ ]:
from rdkit.ML.Scoring import Scoring

def _metrics_from_sim(sim, labels_rest):
    order = np.argsort(sim)[::-1]
    scores = np.column_stack([np.asarray(sim)[order], np.asarray(labels_rest)[order]])
    ef = Scoring.CalcEnrichment(scores, 1, [0.01])
    bedroc = Scoring.CalcBEDROC(scores, 1, 20)
    return {"EF1%": float(ef[0]), "BEDROC": float(bedroc)}

def retrieve_simfn(descs, fns, query_ids, sim_fn):
    labels = np.asarray([1 if f[0] == "l" else 0 for f in fns])
    stems = [f[:-6] if f.endswith(".npy") else f.rsplit(".", 1)[0] for f in fns]
    rows = []
    for qid in query_ids:
        hits = [j for j, s in enumerate(stems) if s == qid or s.endswith(qid)]
        if not hits:
            log.warning("  query '%s' not found; skipping", qid); continue
        i = hits[0]
        rest = [labels[z] for z in range(len(fns)) if z != i]
        sim = sim_fn(descs, i)
        m = _metrics_from_sim(sim, rest); m["RefMol"] = stems[i]; rows.append(m)
    return pd.DataFrame(rows, columns=["RefMol", "EF1%", "BEDROC"])

## 4 · Encoders
Each returns `(descs, sim_fn, fns)`. GM-MolSG fuses per-block cosine; GE-MolSG uses chi²; shape/ESP baselines use their own pairwise scores.

In [ ]:
import ge_molsg as gm
from sklearn.metrics.pairwise import chi2_kernel
import exp3_methods as M
from vlad import vlad_bof

# ── GE-MolSG (standalone geo) ────────────────────────────────────────────────
def encode_ge_molsg(esp_dir, target_root, fns):
    codebook = np.load(GE_CODEBOOK, allow_pickle=True)
    vecs = []
    for f in fns:
        mol = np.load(str(esp_dir / f), allow_pickle=True)
        coords, esp = mol[0], mol[2]
        feat = np.concatenate([coords, (esp * EW).reshape(-1, 1)], axis=1)
        wks = np.nan_to_num(M.geo_wks_ge(feat, K, EVALS, VAR, KNN, LAP_NORM))
        vecs.append(M.knn_histogram(wks, codebook, knn=BOF_KNN))
    V = np.asarray(vecs)
    S = chi2_kernel(V)
    def sim_fn(_d, i):
        keep = np.arange(S.shape[0]) != i
        return S[i][keep]
    return V, sim_fn, fns

# ── GM-MolSG (fused geo BoF + patch VLAD) ────────────────────────────────────
def encode_gm_molsg(esp_dir, target_root, fns):
    from rdkit import Chem
    geo_cb = np.load(GM_GEO_CODEBOOK, allow_pickle=True)
    patch_cb = np.load(GM_PATCH_CODEBOOK, allow_pickle=True)
    patch_fts = M.parse_feature_list(PATCH_FTS, M.VALID_PATCH_FEATURES, "patch")
    pdb_dir = target_root / "PDB_Files"

    geo_vecs, vlad_vecs, keep = [], [], []
    for f in fns:
        stem = f[:-4]
        mol = np.load(str(esp_dir / f), allow_pickle=True)
        coords, faces, esp = mol[0], mol[1], mol[2]
        feat = np.concatenate([coords, (esp * EW).reshape(-1, 1)], axis=1)
        wks = np.nan_to_num(M.geo_wks_gm(feat, K, EVALS, VAR, KNN, LAP_NORM))
        geo = M.knn_histogram(wks, geo_cb, knn=BOF_KNN)
        # patch VLAD block
        pdb = pdb_dir / f"{stem}.pdb"
        rdmol = Chem.rdmolfiles.MolFromPDBFile(str(pdb), removeHs=False) if pdb.exists() else None
        if rdmol is None:
            log.warning("  [GM-MolSG] missing PDB %s; skipping", pdb); continue
        patch_feat = M.build_patch_feat(
            coords, faces, esp, rdmol, patch_fts, elec_weight=EW,
            logp_sigma=LOGP_SIGMA, logp_radius=LOGP_RADIUS,
            jazzy_sigma=JAZZY_SIGMA, jazzy_radius=JAZZY_RADIUS)
        from patches import compute_patches
        patch_desc = compute_patches(coords, patch_feat, radius=PATCH_RADII, normalise=PATCH_NORM)
        vlad = vlad_bof([patch_desc, patch_cb, None], soft=False)
        geo_vecs.append(geo); vlad_vecs.append(vlad); keep.append(f)

    geo_block = np.asarray(geo_vecs)
    vlad_block = np.asarray(vlad_vecs)
    S_geo = M.cosine_matrix(geo_block)
    S_vlad = M.cosine_matrix(vlad_block)
    S = M.linear_fusion(S_geo, S_vlad, ALPHA)
    descs = list(range(len(keep)))   # index proxy; sim taken from S
    def sim_fn(_d, i):
        keep_m = np.arange(S.shape[0]) != i
        return S[i][keep_m]
    return descs, sim_fn, keep

# ── ElectroShape / ElectroShape5D (MMFF94) ───────────────────────────────────
_CHARGE_SCALE, _ALOGP_SCALE = 25.0, 5.0

def _usr_moments(d):
    mu = np.mean(d); sig = np.std(d); m3 = np.mean((d - mu) ** 3)
    return np.array([mu, sig, np.cbrt(m3)])

def _electroshape5d(pbmol, rdmol):
    from rdkit.Chem import Crippen
    coords = np.array([a.coords for a in pbmol.atoms], dtype=np.float64)
    charges = np.array([a.partialcharge for a in pbmol.atoms], dtype=np.float64)
    if rdmol is None:
        alogp = np.zeros(len(coords))
    else:
        contribs = Crippen._GetAtomContribs(rdmol)
        if len(contribs) == len(coords):
            alogp = np.array([c[0] for c in contribs], dtype=np.float64)
        else:
            heavy = [a.GetIdx() for a in rdmol.GetAtoms() if a.GetAtomicNum() != 1]
            alogp = (np.array([contribs[i][0] for i in heavy], dtype=np.float64)
                     if len(heavy) == len(coords) else np.zeros(len(coords)))
    pts = np.column_stack([coords, charges * _CHARGE_SCALE, alogp * _ALOGP_SCALE])
    dto = lambda p: np.linalg.norm(pts - p, axis=1)
    c1 = pts.mean(0); c2 = pts[np.argmax(dto(c1))]; c3 = pts[np.argmax(dto(c2))]
    c4 = pts[np.argmax(dto((c1 + c2 + c3) / 3.0))]
    c5 = pts[np.argmax(dto((c1 + c2 + c3 + c4) / 4.0))]
    c6 = c1.copy(); c6[4] += np.linalg.norm(c1 - c2)
    return np.concatenate([_usr_moments(dto(c)) for c in (c1, c2, c3, c4, c5, c6)])

def _read_pdb_ob(fp):
    from openbabel import openbabel as ob
    from openbabel import pybel
    mol = ob.OBMol(); conv = ob.OBConversion(); conv.SetInFormat("pdb")
    conv.ReadFile(mol, str(fp))
    cm = ob.OBChargeModel.FindType(ESHAPE_CHARGE_MODEL); cm.ComputeCharges(mol)
    return pybel.Molecule(mol)

def _encode_eshape(esp_dir, target_root, fns, five_d):
    from oddt.shape import electroshape, usr_similarity
    from rdkit import Chem
    pdb_dir = target_root / "PDB_Files"
    descs, keep = [], []
    for f in fns:
        stem = f[:-4]; pdb = pdb_dir / f"{stem}.pdb"
        if not pdb.exists():
            log.warning("  [ElectroShape] missing PDB %s", pdb); descs.append(None); keep.append(f); continue
        try:
            pb = _read_pdb_ob(pdb)
            if five_d:
                rd = Chem.rdmolfiles.MolFromPDBFile(str(pdb), removeHs=False, sanitize=True)
                descs.append(_electroshape5d(pb, rd))
            else:
                descs.append(electroshape(pb))
        except Exception as e:
            log.warning("  [ElectroShape] %s failed: %r", stem, e); descs.append(None)
        keep.append(f)
    if five_d:
        def sim_fn(d, i):
            ref = np.asarray(d[i], dtype=np.float64) if d[i] is not None else None
            out = []
            for z in range(len(d)):
                if z == i: continue
                if ref is None or d[z] is None:
                    out.append(0.0)
                else:
                    b = np.asarray(d[z], dtype=np.float64)
                    out.append(1.0 / (1.0 + np.sum(np.abs(ref - b)) / len(ref)))
            return np.asarray(out)
    else:
        def sim_fn(d, i):
            ref = d[i]
            return np.asarray([usr_similarity(ref, d[z]) if (ref is not None and d[z] is not None) else 0.0
                               for z in range(len(d)) if z != i])
    return descs, sim_fn, keep

def encode_electroshape(esp_dir, target_root, fns):
    return _encode_eshape(esp_dir, target_root, fns, five_d=False)

def encode_electroshape5d(esp_dir, target_root, fns):
    return _encode_eshape(esp_dir, target_root, fns, five_d=True)

# ── ESP-Sim (ML-charge) ──────────────────────────────────────────────────────
def encode_espsim(esp_dir, target_root, fns):
    from copy import deepcopy
    from rdkit import Chem
    from rdkit.Chem import rdMolAlign, rdMolDescriptors
    from espsim import GetShapeSim, GetEspSim
    from espsim.helpers import mlCharges
    pdb_dir = target_root / "PDB_Files"
    descs, keep = [], []
    for f in fns:
        stem = f[:-4]; pdb = pdb_dir / f"{stem}.pdb"
        m = Chem.rdmolfiles.MolFromPDBFile(str(pdb), removeHs=False, sanitize=True) if pdb.exists() else None
        if m is None:
            log.warning("  [ESP-Sim] missing/unparseable %s", stem); descs.append(None); keep.append(f); continue
        try:
            descs.append([m, mlCharges([m])])
        except Exception as e:
            log.warning("  [ESP-Sim] charge calc failed %s: %r", stem, e); descs.append(None)
        keep.append(f)
    def _align(prb, ref):
        pC = rdMolDescriptors._CalcCrippenContribs(prb)
        rC = rdMolDescriptors._CalcCrippenContribs(ref)
        rdMolAlign.GetCrippenO3A(prb, ref, pC, rC, 0, 0).Align()
    def _score(pe, re):
        if pe is None or re is None: return 0.0
        prb = deepcopy(pe[0]); ref = deepcopy(re[0])
        try:
            _align(prb, ref); shape = GetShapeSim(prb, ref)
            esp = GetEspSim(prb, ref, 0, 0, prbCharge=pe[1], refCharge=re[1],
                            renormalize=True, metric="tanimoto")
            return shape + esp
        except Exception:
            return 0.0
    def sim_fn(d, i):
        ref = d[i]
        return np.asarray([_score(d[z], ref) for z in range(len(d)) if z != i])
    return descs, sim_fn, keep

# ── Roshambo (Roshambo2 combination score) ───────────────────────────────────
def encode_roshambo(esp_dir, target_root, fns):
    from rdkit import Chem
    pdb_dir = target_root / "PDB_Files"
    mols, keep = [], []
    for f in fns:
        stem = f[:-4]; pdb = pdb_dir / f"{stem}.pdb"
        m = Chem.rdmolfiles.MolFromPDBFile(str(pdb), removeHs=False, sanitize=True) if pdb.exists() else None
        if m is None:
            log.warning("  [Roshambo] missing/unparseable %s", stem); continue
        m.SetProp("_Name", stem); mols.append(m); keep.append(f)
    def sim_fn(d, i):
        from roshambo2 import Roshambo2
        ref = d[i]; others = [d[z] for z in range(len(d)) if z != i]
        try:
            calc = Roshambo2(ref, others, color=True)
            raw = calc.compute(backend="cpp", reduce_over_conformers=True,
                               optim_mode="combination", write_scores=False)
            df = raw[list(raw.keys())[0]]
            pos = {mol.GetProp("_Name"): p for p, mol in enumerate(others)}
            arr = np.zeros(len(others))
            for _, row in df.iterrows():
                p = pos.get(row["name"])
                if p is not None:
                    arr[p] = row["tanimoto_combo_legacy"]
            return arr
        except Exception as e:
            log.warning("  [Roshambo] ref %d failed: %r", i, e)
            return np.zeros(len(others))
    return mols, sim_fn, keep

ENCODERS = {
    "GM-MolSG": encode_gm_molsg,
    "GE-MolSG": encode_ge_molsg,
    "ElectroShape": encode_electroshape,
    "ElectroShape5D": encode_electroshape5d,
    "ESP-Sim": encode_espsim,
    "Roshambo": encode_roshambo,
}

## 5 · Per-target driver (iterative, cached)

In [ ]:
def run_target(target):
    out_by_method = {}
    esp_dir = target_root = base_fns = query_ids = None
    for method in METHODS:
        csv_path = RESULTS_ROOT / method / f"{target}.csv"
        if csv_path.exists() and not FORCE:
            log.info("[%s | %s] cached -> %s", target, method, csv_path)
            out_by_method[method] = pd.read_csv(csv_path); continue
        if esp_dir is None:
            esp_dir, target_root = resolve_target_dir(target, DATA_ROOT, OUT_DIR / "_work")
            base_fns = list_surface_fns(esp_dir)
            query_ids = load_query_ids(resolve_queries_csv(QUERIES_DIR, target))
            n_lig = sum(f[0] == "l" for f in base_fns)
            log.info("[%s] %d surfaces (%d ligands, %d decoys), %d queries",
                     target, len(base_fns), n_lig, len(base_fns) - n_lig, len(query_ids))
        log.info("[%s | %s] encoding ...", target, method)
        try:
            descs, sim_fn, fns = ENCODERS[method](esp_dir, target_root, base_fns)
        except Exception as e:
            log.error("[%s | %s] encoding failed: %r", target, method, e)
            out_by_method[method] = pd.DataFrame(columns=["RefMol", "EF1%", "BEDROC"]); continue
        log.info("[%s | %s] retrieval ...", target, method)
        df = retrieve_simfn(descs, fns, query_ids, sim_fn)
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(csv_path, index=False)
        log.info("[%s | %s] %d queries  mean EF1%%=%.3f  mean BEDROC=%.3f -> %s",
                 target, method, len(df),
                 df["EF1%"].mean() if len(df) else float("nan"),
                 df["BEDROC"].mean() if len(df) else float("nan"), csv_path)
        out_by_method[method] = df
    return out_by_method

sampled_data = {m: {} for m in METHODS}
for ti, target in enumerate(TARGETS, 1):
    log.info("==== target %d/%d : %s ====", ti, len(TARGETS), target)
    try:
        by_method = run_target(target)
    except FileNotFoundError as e:
        log.error("skipping %s: %s", target, e); continue
    for method, df in by_method.items():
        if len(df):
            sampled_data[method][target] = df
log.info("encoding + retrieval complete")

## 6 · Summary

In [ ]:
rows = []
for method in sampled_data:
    for t in TARGETS:
        if t in sampled_data[method] and len(sampled_data[method][t]):
            df = sampled_data[method][t]
            rows.append(dict(method=method, target=t, mean_EF1=df["EF1%"].mean(),
                             mean_BEDROC=df["BEDROC"].mean(), n_queries=len(df)))
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "summary_per_target.csv", index=False)
log.info("wrote summary_per_target.csv (%d rows)", len(summary))
summary

## 7 · Scatter / line plots — BEDROC and EF1%

In [ ]:
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid", font_scale=1.0)
except Exception:
    pass
COLOUR_MAP = {"GM-MolSG":"#2CA02C","GE-MolSG":"#0072B2","ElectroShape":"#FF6B6B",
              "ElectroShape5D":"#D55E00","ESP-Sim":"#E69F00","Roshambo":"#CC79A7"}
METHOD_MARKERS = {"GM-MolSG":"o","GE-MolSG":"s","ElectroShape":"v",
                  "ElectroShape5D":"D","ESP-Sim":"^","Roshambo":"*"}
SCATTER_METHODS = METHODS

def scatter(metric):
    tsorted = sorted(TARGETS); x = np.arange(len(tsorted))
    y_max = 50.0 if metric == "EF1%" else 0.9
    fig, ax = plt.subplots(figsize=(max(8, len(tsorted)*0.35), 4.5), facecolor="white")
    ax.set_facecolor("#EBEBEB")
    ax.grid(axis="y", linestyle="-", color="white", linewidth=0.8, zorder=0)
    ax.grid(axis="x", linestyle="-", color="white", linewidth=0.8, zorder=0)
    ax.margins(x=0.02)
    for method in SCATTER_METHODS:
        means = [sampled_data[method][t][metric].mean()
                 if t in sampled_data.get(method, {}) and len(sampled_data[method][t]) else np.nan
                 for t in tsorted]
        col = COLOUR_MAP.get(method, "#888888")
        ax.plot(x, means, color=col, linewidth=1.4, marker=METHOD_MARKERS.get(method,"o"),
                markersize=6, markerfacecolor="white", markeredgecolor=col,
                markeredgewidth=1.5, label=method, zorder=3)
    ax.set_xlim(-0.5, len(tsorted)-0.5); ax.set_ylim(0.0, y_max)
    ax.set_xticks(x); ax.set_xticklabels(tsorted, rotation=90, ha="center", fontsize=9)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_ylabel(metric, fontsize=12); ax.set_xlabel("Target", fontsize=12)
    title = {"BEDROC":"DUDE-Z Mean BEDROC (α=20) Retrieval Performance",
             "EF1%":"DUDE-Z Mean EF1% Retrieval Performance"}.get(metric, f"DUDE-Z Mean {metric}")
    ax.set_title(title, fontsize=14, pad=6)
    ax.legend(fontsize=9, bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0.0, framealpha=0.85)
    plt.tight_layout()
    path = OUT_DIR / f"scatter_{metric.replace('%','pct')}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.show(); log.info("saved %s", path)

scatter("BEDROC"); scatter("EF1%")

## 8 · Wilcoxon signed-rank test — focused on GM-MolSG

In [ ]:
from scipy.stats import wilcoxon
WILCOXON_METHOD = "GM-MolSG"; alpha = 0.05; records = []
for metric in METRICS:
    for baseline in sampled_data:
        if baseline == WILCOXON_METHOD: continue
        pf, pb = [], []
        for t in TARGETS:
            if (t in sampled_data[WILCOXON_METHOD] and len(sampled_data[WILCOXON_METHOD].get(t, [])) and
                    t in sampled_data[baseline] and len(sampled_data[baseline].get(t, []))):
                pf.append(sampled_data[WILCOXON_METHOD][t][metric].mean())
                pb.append(sampled_data[baseline][t][metric].mean())
        n = len(pf)
        if n < 4:
            records.append(dict(Metric=metric, Baseline=baseline, N_targets=n, W=np.nan,
                                p_value=np.nan, Significant="—", Direction="insufficient data")); continue
        diffs = np.array(pf) - np.array(pb)
        if np.all(diffs == 0):
            records.append(dict(Metric=metric, Baseline=baseline, N_targets=n, W=np.nan,
                                p_value=np.nan, Significant="—", Direction="identical")); continue
        stat, p = wilcoxon(pf, pb, alternative="two-sided")
        records.append(dict(Metric=metric, Baseline=baseline, N_targets=n, W=round(stat,3),
                            p_value=round(p,4), Significant="✓" if p < alpha else "✗",
                            Direction="better" if diffs.mean() > 0 else "worse"))
sig_df = pd.DataFrame(records)
if len(sig_df):
    sig_df = sig_df.sort_values(["Metric", "p_value"])
sig_df.to_csv(OUT_DIR / f"wilcoxon_{WILCOXON_METHOD.replace(' ','_')}.csv", index=False)
print(f"Wilcoxon signed-rank test: {WILCOXON_METHOD} vs all others (α={alpha})\n")
for metric in METRICS:
    if "Metric" not in sig_df.columns: break
    sub = sig_df[sig_df.Metric == metric][["Baseline","N_targets","W","p_value","Significant","Direction"]]
    print(f"── {metric} {'─'*40}"); print(sub.to_string(index=False)); print()
sig_df